# Training Loop: From Data to Decision Boundaries

**A complete training pipeline using the grilly framework.**

This notebook covers:
1. GPU/CPU detection
2. Generating a synthetic spiral dataset (2D, 3 classes)
3. Building a 3-layer MLP for classification
4. DataLoader with mini-batch training
5. Training loop: forward -> loss -> backward -> optimizer step
6. Plotting the loss curve and decision boundary
7. Saving and loading a checkpoint

All plots use matplotlib. This notebook is self-contained and runs on both
GPU (Vulkan) and CPU (numpy fallback).

In [ ]:
import grilly
import grilly.functional as F
import matplotlib.pyplot as plt
import numpy as np
from grilly import nn
from grilly.nn import Variable
from grilly.optim import AdamW
from grilly.utils import DataLoader, TensorDataset

# --- Backend detection ---
try:
    from grilly._bridge import is_vulkan_available
    DEVICE = "vulkan" if is_vulkan_available() else "cpu"
except (ImportError, AttributeError):
    DEVICE = "cpu"

print(f"grilly {grilly.__version__} | backend: {DEVICE}")
np.random.seed(42)

## 1. Synthetic Spiral Dataset

We generate a 3-class spiral dataset in 2D. Each class forms a spiral arm
radiating from the origin. This is a classic nonlinear classification problem
that requires hidden layers to solve.

In [ ]:
def make_spiral_data(n_points=300, n_classes=3, noise=0.15):
    """Generate a 2D spiral dataset with n_classes arms."""
    X = np.zeros((n_points * n_classes, 2), dtype=np.float32)
    y = np.zeros(n_points * n_classes, dtype=np.int64)

    for c in range(n_classes):
        ix = range(n_points * c, n_points * (c + 1))
        # Radius grows linearly, angle spans ~2 full turns per class
        r = np.linspace(0.0, 1.0, n_points)
        theta = np.linspace(c * 4.0, (c + 1) * 4.0, n_points) + np.random.randn(n_points) * noise
        X[ix] = np.column_stack([r * np.sin(theta), r * np.cos(theta)])
        y[ix] = c

    return X, y

X_train, y_train = make_spiral_data(n_points=300, n_classes=3)

print(f"Training data : X={X_train.shape}, y={y_train.shape}")
print(f"Classes       : {np.unique(y_train)}")

# Visualize the dataset
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71']
for c in range(3):
    mask = y_train == c
    ax.scatter(X_train[mask, 0], X_train[mask, 1], c=colors[c],
               s=10, alpha=0.7, label=f'Class {c}')
ax.set_title('Spiral Dataset (3 classes)')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 2. DataLoader Setup

Wrap our numpy arrays into a `TensorDataset` and create a `DataLoader` for
mini-batch iteration. This handles shuffling and batching automatically.

In [ ]:
# One-hot encode labels for cross-entropy via softmax + MSE
n_classes = 3
y_onehot = np.zeros((len(y_train), n_classes), dtype=np.float32)
y_onehot[np.arange(len(y_train)), y_train] = 1.0

# Create DataLoader
dataset = TensorDataset(X_train, y_onehot)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

print(f"Dataset size : {len(dataset)}")
print("Batch size   : 64")
print(f"Batches/epoch: {len(dataset) // 64 + (1 if len(dataset) % 64 else 0)}")

## 3. Build the Model

A 3-layer MLP with ReLU activations. We use small dimensions (64, 64) since
our data is only 2D -- this keeps things fast and avoids OOM.

```
Input (2) -> Linear(2, 64) -> ReLU -> Linear(64, 64) -> ReLU -> Linear(64, 3) -> Output (3)
```

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, n_classes),
)

optimizer = AdamW(model.parameters(), lr=1e-2, weight_decay=1e-4)

total_params = sum(p.data.size for p in model.parameters())
print("Model: 2 -> 64 -> 64 -> 3")
print(f"Total parameters: {total_params:,}")

## 4. Training Loop

We train for 200 epochs. Each epoch iterates over all mini-batches:

1. **Forward**: compute logits and softmax probabilities
2. **Loss**: cross-entropy approximated as MSE(softmax(logits), one_hot)
3. **Backward**: compute gradients via autograd
4. **Step**: update parameters with AdamW

In [ ]:
n_epochs = 200
loss_history = []

for epoch in range(n_epochs):
    epoch_loss = 0.0
    n_batches = 0

    for X_batch, y_batch in dataloader:
        # Wrap numpy arrays as Variables for autograd
        x_var = Variable(X_batch)
        t_var = Variable(y_batch)

        # Forward pass
        logits = model(x_var)

        # Softmax + MSE loss (simple cross-entropy surrogate)
        probs = F.softmax(logits)
        diff = probs - t_var
        loss = (diff * diff).sum() / X_batch.shape[0]

        # Backward pass
        loss.backward()

        # Optimizer step
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += float(loss.data)
        n_batches += 1

    avg_loss = epoch_loss / max(n_batches, 1)
    loss_history.append(avg_loss)

    if (epoch + 1) % 50 == 0 or epoch == 0:
        # Compute accuracy on full dataset
        full_logits = model(Variable(X_train))
        preds = np.argmax(full_logits.data, axis=1)
        acc = np.mean(preds == y_train) * 100
        print(f"Epoch {epoch+1:3d}/{n_epochs} | Loss: {avg_loss:.4f} | Accuracy: {acc:.1f}%")

print("\nTraining complete.")

## 5. Plot Loss Curve

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(loss_history, linewidth=1.5, color='#2c3e50')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MSE on softmax)')
ax.set_title('Training Loss Curve')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Visualize Decision Boundary

We evaluate the model on a dense grid of 2D points and color each point
by its predicted class. The scatter plot overlays the training data.

In [ ]:
def plot_decision_boundary(model, X, y, resolution=200):
    """Plot the decision boundary of a 2D classifier."""
    x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
    y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, resolution),
        np.linspace(y_min, y_max, resolution)
    )

    grid = np.column_stack([xx.ravel(), yy.ravel()]).astype(np.float32)
    logits = model(Variable(grid))
    preds = np.argmax(logits.data, axis=1).reshape(xx.shape)

    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    ax.contourf(xx, yy, preds, levels=[-0.5, 0.5, 1.5, 2.5],
                colors=['#fadbd8', '#d4efdf', '#d6eaf8'], alpha=0.6)
    ax.contour(xx, yy, preds, levels=[0.5, 1.5], colors='gray',
               linewidths=1.0, linestyles='--')

    colors = ['#e74c3c', '#2ecc71', '#3498db']
    for c in range(3):
        mask = y == c
        ax.scatter(X[mask, 0], X[mask, 1], c=colors[c],
                   s=12, alpha=0.8, edgecolors='white', linewidths=0.3,
                   label=f'Class {c}')

    ax.set_title('Learned Decision Boundary')
    ax.legend(loc='upper right')
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()

plot_decision_boundary(model, X_train, y_train)

## 7. Save and Load Checkpoint

Grilly supports saving model state dictionaries as numpy `.npz` files.
This lets you resume training or deploy a trained model later.

In [ ]:
import os
import tempfile

# --- Save checkpoint ---
checkpoint_path = os.path.join(tempfile.gettempdir(), "spiral_model.npz")

# Collect all parameter arrays into a dict
state_dict = {}
for i, param in enumerate(model.parameters()):
    state_dict[f"param_{i}"] = param.data

np.savez(checkpoint_path, **state_dict)
print(f"Checkpoint saved to: {checkpoint_path}")
print(f"File size: {os.path.getsize(checkpoint_path) / 1024:.1f} KB")

# --- Load checkpoint ---
loaded = np.load(checkpoint_path)
for i, param in enumerate(model.parameters()):
    param.data = loaded[f"param_{i}"]

print("\nCheckpoint loaded successfully.")

# Verify: run inference after loading
logits_check = model(Variable(X_train))
preds_check = np.argmax(logits_check.data, axis=1)
acc_check = np.mean(preds_check == y_train) * 100
print(f"Post-load accuracy: {acc_check:.1f}%")

## Summary

In this notebook you learned:

- **Synthetic data generation**: spiral datasets for nonlinear classification
- **DataLoader**: batching and shuffling with `TensorDataset` and `DataLoader`
- **Training loop**: the forward -> loss -> backward -> step cycle
- **Visualization**: loss curves and decision boundaries with matplotlib
- **Checkpointing**: save/load model parameters as `.npz` files

### Next Steps

- **Notebook 03**: Spiking neural networks with LIF neurons
- **Notebook 04**: Vector symbolic architectures
- **Notebook 05**: Attention mechanisms and transformers